# CrowdShield CV Pipeline Colab Smoke Test

This notebook installs the runtime dependencies, stages the cv_pipeline modules, runs `CVPipeline.process_video()` on a test clip, and renders a visual sanity-check overlay for detections and zone densities.

In [ ]:
!pip install -q ultralytics opencv-python-headless tqdm matplotlib

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

from google.colab import drive, files

USE_DRIVE_FOR_VIDEO = False
USE_DRIVE_FOR_MODULES = True

VIDEO_PATH = None

if USE_DRIVE_FOR_VIDEO or USE_DRIVE_FOR_MODULES:
    drive.mount('/content/drive')

if USE_DRIVE_FOR_VIDEO:
    VIDEO_PATH = Path('/content/drive/MyDrive/CrowdShield/test_videos/your_test_video.mp4')
    if not VIDEO_PATH.exists():
        raise FileNotFoundError(f'Drive video not found: {VIDEO_PATH}')
else:
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No video was uploaded.')
    uploaded_name = next(iter(uploaded))
    VIDEO_PATH = Path('/content') / uploaded_name
    with VIDEO_PATH.open('wb') as handle:
        handle.write(uploaded[uploaded_name])

WORK_ROOT = Path('/content/crowdshield_cv_pipeline')
SCRIPTS_DIR = WORK_ROOT / 'scripts'
MODELS_DIR = WORK_ROOT / 'models'
SCRIPTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
(MODELS_DIR / '__init__.py').write_text('', encoding='utf-8')

REQUIRED_SCRIPTS = ['detector.py', 'optical_flow.py', 'tracker.py', 'zone_config.py', 'pipeline.py']
SOURCE_CV_PIPELINE_DIR = Path('/content/drive/MyDrive/CrowdShield/ai-core/cv_pipeline')
REPO_URL = ''  # Optional: set this if you prefer cloning the repo instead of copying from Drive.

if USE_DRIVE_FOR_MODULES and SOURCE_CV_PIPELINE_DIR.exists():
    source_scripts = SOURCE_CV_PIPELINE_DIR / 'scripts'
    source_models = SOURCE_CV_PIPELINE_DIR / 'models'
    for module_name in REQUIRED_SCRIPTS:
        shutil.copy2(source_scripts / module_name, SCRIPTS_DIR / module_name)
    shutil.copy2(source_models / 'download_weights.py', MODELS_DIR / 'download_weights.py')
elif REPO_URL:
    repo_root = Path('/content/CrowdShield')
    if not repo_root.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(repo_root)], check=True)
    source_root = repo_root / 'ai-core' / 'cv_pipeline'
    source_scripts = source_root / 'scripts'
    source_models = source_root / 'models'
    for module_name in REQUIRED_SCRIPTS:
        shutil.copy2(source_scripts / module_name, SCRIPTS_DIR / module_name)
    shutil.copy2(source_models / 'download_weights.py', MODELS_DIR / 'download_weights.py')
else:
    raise FileNotFoundError(
        'Set USE_DRIVE_FOR_MODULES=True with a mounted Drive repo path, or fill REPO_URL to clone the repo.'
    )

if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

print('Video path:', VIDEO_PATH)
print('Module root:', WORK_ROOT)
print('Scripts path:', SCRIPTS_DIR)
print('Models path:', MODELS_DIR)

In [ ]:
import json
import time

import cv2
import numpy as np
from tqdm.auto import tqdm

from pipeline import CVPipeline
from zone_config import generate_grid_zones

zones = generate_grid_zones(3, 3)
pipeline = CVPipeline(VIDEO_PATH, zones)

capture = cv2.VideoCapture(str(VIDEO_PATH))
total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
capture.release()

start_time = time.perf_counter()
records = []
for record in tqdm(
    pipeline.process_video(
        video_path=VIDEO_PATH,
        sample_every_n_frames=1,
        mode='stream',
    ),
    total=total_frames if total_frames > 0 else None,
    desc='Processing frames',
):
    records.append(record)
elapsed_seconds = time.perf_counter() - start_time

total_processed_frames = len(records)
avg_processing_fps = total_processed_frames / elapsed_seconds if elapsed_seconds > 0 else 0.0
avg_crowd_count_across_zones = (
    sum(zone['crowd_count'] for record in records for zone in record['zones'])
    / (total_processed_frames * len(zones))
    if total_processed_frames and zones
    else 0.0
)
flow_rows = [
    {
        'frame_number': record['frame_number'],
        'timestamp': record['timestamp'],
        'zone_id': zone['zone_id'],
        'crowd_count': zone['crowd_count'],
        'avg_flow_speed': zone['avg_flow_speed'],
        'avg_flow_direction_deg': zone['avg_flow_direction_deg'],
        'avg_flow_direction_label': zone['avg_flow_direction_label'],
    }
    for record in records
    for zone in record['zones']
]
anomaly_frames = []
for record in records:
    flagged_zones = [
        {
            'zone_id': zone['zone_id'],
            'anomaly_flags': zone['anomaly_flags'],
        }
        for zone in record['zones']
        if zone['anomaly_flags']
    ]
    if flagged_zones:
        anomaly_frames.append({
            'frame_number': record['frame_number'],
            'zones': flagged_zones,
        })

print('Summary stats')
print('--------------')
print(f'Total frames processed: {total_processed_frames}')
print(f'Average processing FPS: {avg_processing_fps:.2f}')
print(f'Average crowd count across all zones: {avg_crowd_count_across_zones:.3f}')
print(f'Collected zone flow rows: {len(flow_rows)}')
print()
print('Sample zone flow metrics')
print('------------------------')
print(json.dumps(flow_rows[:20], indent=2))
print()
print('Average flow metrics by zone')
print('----------------------------')
zone_flow_summary = []
for zone in zones:
    zone_rows = [row for row in flow_rows if row['zone_id'] == zone.zone_id]
    if zone_rows:
        avg_speed = sum(row['avg_flow_speed'] for row in zone_rows) / len(zone_rows)
        label_counts = {}
        for row in zone_rows:
            label_counts[row['avg_flow_direction_label']] = label_counts.get(row['avg_flow_direction_label'], 0) + 1
        dominant_label = max(label_counts, key=label_counts.get)
    else:
        avg_speed = 0.0
        dominant_label = 'N'
    zone_flow_summary.append({
        'zone_id': zone.zone_id,
        'avg_flow_speed': round(avg_speed, 4),
        'dominant_flow_direction_label': dominant_label,
    })
print(json.dumps(zone_flow_summary, indent=2))
print()
print('Frames with non-empty anomaly flags:')
print(json.dumps(anomaly_frames[:20], indent=2))

## Visual sanity-check

This cell draws detection boxes, zone grid lines, and per-zone density labels on a single frame so you can verify the JSON output is grounded in the video content.

In [ ]:
import matplotlib.pyplot as plt

from detector import CrowdDetector

detector = CrowdDetector()
preview_capture = cv2.VideoCapture(str(VIDEO_PATH))
preview_frame_index = 30 if total_frames == 0 else min(30, max(total_frames - 1, 0))
preview_capture.set(cv2.CAP_PROP_POS_FRAMES, preview_frame_index)
ok, frame = preview_capture.read()
preview_capture.release()
if not ok:
    raise RuntimeError('Could not read a preview frame from the uploaded video.')

frame_height, frame_width = frame.shape[:2]
detections = detector.detect_frame(frame)
zone_assignments = detector.assign_to_zones(detections, zones, frame_width, frame_height)
overlay = frame.copy()

for zone in zones:
    bounds = zone.bounds_normalized
    x1 = int(bounds['x_min'] * frame_width)
    y1 = int(bounds['y_min'] * frame_height)
    x2 = int(bounds['x_max'] * frame_width)
    y2 = int(bounds['y_max'] * frame_height)
    density = detector.compute_density(zone_assignments.get(zone.zone_id, []), zone)
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 255), 2)
    label = f'{zone.zone_id} dens={density:.2f}'
    cv2.putText(overlay, label, (x1 + 6, y1 + 22), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 0), 2, cv2.LINE_AA)

for det in detections:
    x1, y1, x2, y2 = [int(value) for value in det['bbox']]
    cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 200, 0), 2)
    cv2.circle(overlay, (int(det['center'][0]), int(det['center'][1])), 4, (0, 0, 255), -1)

overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(16, 9))
plt.imshow(overlay_rgb)
plt.axis('off')
plt.title(f'Preview frame {preview_frame_index} with detections and 3x3 zone grid')
plt.show()